## Step 1: Setup (Create table movies_dups)

In [ ]:
USE DATABASE movies_db;
USE schema movies_schema;

CREATE OR REPLACE TABLE movies_dups AS
SELECT id, title, release_date
FROM movies, VALUES (1), (1);

SELECT *
FROM movies_dups;

## Step 2: Strategy 1 - Add a new column: row_id

In [ ]:
ALTER TABLE movies_dups ADD COLUMN row_id STRING;

UPDATE movies_dups
SET row_id = UUID_STRING();

SELECT *
FROM movies_dups;

## Step 3: Strategy #1 - Delete duplicates using MIN(row_id)

In [ ]:
DELETE FROM movies_dups a
WHERE row_id > (SELECT MIN(row_id) 
                FROM movies_dups b 
                WHERE b.id = a.id
                  AND b.title = a.title
                  AND b.release_date = a.release_date
                );

ALTER TABLE movies_dups
DROP COLUMN row_id;

SELECT *
FROM movies_dups;

## Step 3: Strategy #2 - Delete duplicates using ROW_NUMBER

In [ ]:
DELETE FROM movies_dups
WHERE (ROW_ID, 1) NOT IN (SELECT ROW_ID, ROW_NUMBER() OVER(PARTITION BY id ORDER BY row_id)
                          FROM movies_dups
                         );

ALTER TABLE movies_dups
DROP COLUMN row_id;

SELECT *
FROM movies_dups;

## Step2: Strategy 2 - Swap Tables 

In [ ]:
CREATE OR REPLACE TABLE movies_dups2 LIKE movies_dups COPY GRANTS;
INSERT INTO movies_dups2 SELECT DISTINCT * FROM movies_dups;
ALTER TABLE movies_dups SWAP WITH movies_dups2;
DROP TABLE movies_dups2;
SELECT * FROM movies_dups;

## Step2: Strategy 3 - INSERT OVERWRITE

In [ ]:
INSERT OVERWRITE INTO movies_dups
SELECT DISTINCT * FROM movies_dups;

SELECT * FROM movies_dups;

## Step2: Strategy 4 - FLOW PIPE OPERATOR WITH DELETE AND INSERT

In [ ]:
SELECT DISTINCT * FROM movies_dups
->> DELETE FROM movies_dups
->> INSERT INTO movies_dups SELECT * FROM $2;

SELECT * FROM movies_dups;

## Step2: Strategy 5 - FLOW PIPE OPERATOR WITH TRUNCATE AND INSERT

In [ ]:
SELECT DISTINCT * FROM movies_dups
->> TRUNCATE TABLE movies_dups
->> INSERT INTO movies_dups SELECT * FROM $2;

SELECT * FROM movies_dups;

In [ ]:
INSERT INTO movies_dups
SELECT * FROM movies_dups LIMIT 2;

In [ ]:
CREATE OR REPLACE temporary TABLE temp_movies 
AS
SELECT DISTINCT * 
FROM movies_dups 
QUALIFY COUNT(*) OVER(PARTITION BY id) > 1;

SELECT * FROM temp_movies;

DELETE
FROM movies_dups
WHERE id IN (
    SELECT id
    FROM movies_dups
    GROUP BY id
    HAVING COUNT(*) > 1
);

INSERT INTO movies_dups
SELECT * FROM temp_movies;

SELECT * FROM movies_dups;

